In [ ]:
import os
import shutil
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# 1. АВТОМАТИЧЕСКАЯ РАСПАКОВКА И СОРТИРОВКА ДАТАСЕТА

# Указываем имя вашего архива, который лежит в папке репозитория
zip_file_path = "cats_and_dogs.zip"  
extracted_folder = "./extracted_images"
base_dir = "./cats_and_dogs"

# Проверяем, существует ли папка с готовой структурой. Если нет — распаковываем архив

if not os.path.exists(base_dir):
    print("Архив найден. Начинаю автоматическую распаковку во временную папку...")
    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(extracted_folder)
        
    print("Сканирую файлы и собираю пути к изображениям...")
    all_images = []
    for root_dir, dirs, files in os.walk(extracted_folder):
        for file in files:
            if file.lower().endswith((".png", ".jpg", ".jpeg")):
                all_images.append(os.path.join(root_dir, file))
                
# Разделяем выборку на Обучающую (80%) и Тестовую (20%)
    train_files, test_files = train_test_split(all_images, test_size=0.2, random_state=42)
    
# Создаем целевую структуру директорий для работы корректного ImageFolder
    for path in [f"{base_dir}/train/cats", f"{base_dir}/train/dogs", f"{base_dir}/test/cats", f"{base_dir}/test/dogs"]:
        os.makedirs(path, exist_ok=True)
        
# Функция для безопасного копирования файлов на основе их имен
    def distribute_files(file_list, subfolder):
        for file_path in file_list:
            file_name = os.path.basename(file_path).lower()
            if "cat" in file_name:
                label = "cats"
            elif "dog" in file_name:
                label = "dogs"
            else:
                continue
            shutil.copy(file_path, os.path.join(base_dir, subfolder, label, file_name))

    print("Распределяю фотографии кошек и собак по соответствующим подпапкам...")
    distribute_files(train_files, "train")
    distribute_files(test_files, "test")
    
# Удаляем временный каталог для очистки пространства на диске
    shutil.rmtree(extracted_folder)
    print("Подготовка структуры данных успешно завершена!\n")

In [27]:

# 2. ПАЙПЛАЙН ТРАНСФОРМАЦИЙ 

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Перевод в оттенки серого (1 канал)
    transforms.Resize((32, 32)),                  # Приведение картинок к размеру 32x32 пикселей
    transforms.ToTensor(),                        # Конвертация в тензор PyTorch и масштаб [0, 1]
    transforms.Normalize(mean=[0.5], std=[0.5]),  # Нормализация для ОДНОГО канала
    transforms.Lambda(lambda x: torch.flatten(x)) # Раскатка матрицы пикселей в одномерный плоский вектор
])


In [28]:
# 3. СОЗДАНИЕ ДАТАСЕТОВ И ЗАГРУЗЧИКОВ DATALOADER 

train_dataset = datasets.ImageFolder(root='./cats_and_dogs/train', transform=transform)
test_dataset = datasets.ImageFolder(root='./cats_and_dogs/test', transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

print("Индексы сопоставления классов в системе:", train_dataset.class_to_idx)

Индексы сопоставления классов в системе: {'cats': 0, 'dogs': 1}


In [ ]:
# 4. АРХИТЕКТУРА ОДНОСЛОЙНОГО ПЕРСЕПТРОНА И НАСТРОЙКА

input_features = 1024  # Размер входного вектора (1 канал * 32 пикселя * 32 пикселя)
model = nn.Linear(input_features, 1)  # 1 выходной нейрон для бинарной классификации

criterion = nn.BCEWithLogitsLoss()  # Функция потерь со встроенной стабильной Сигмоидой
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:

# 5. ЦИКЛ ОБУЧЕНИЯ МОДЕЛИ

epochs = 15
print("\n--- Запуск процесса обучения однослойного персептрона ---")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        labels = labels.float().unsqueeze(1)  # Изменение размерности истинных меток до [batch_size, 1]
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Эпоха [{epoch+1}/{epochs}], Текущая ошибка (Loss): {epoch_loss:.4f}")

In [ ]:

# 6. ОЦЕНКА КАЧЕСТВА НА ТЕСТОВОЙ ВЫБОРКЕ

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        probs = torch.sigmoid(outputs)     # Получение вероятностей принадлежности к классу 1
        preds = (probs >= 0.5).int()        # Классификация: >= 0.5 — собака (1), иначе кошка (0)
        
        all_preds.extend(preds.view(-1).tolist())
        all_labels.extend(labels.tolist())

# Расчет финального показателя верности (Accuracy)
correct = sum(p == l for p, l in zip(all_preds, all_labels))
accuracy = correct / len(all_labels)
print(f"\nПоказатель верности (Accuracy) на тестовой выборке: {accuracy * 100:.2f}%")

In [ ]:

# 7. СТРОИТЕЛЬСТВО И ВИЗУАЛИЗАЦИЯ MATРИЦЫ ОШИБОК

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Кошки (0)", "Собаки (1)"],
            yticklabels=["Кошки (0)", "Собаки (1)"])

plt.title("Матрица ошибок (Confusion Matrix)", fontsize=13, fontweight='bold')
plt.ylabel("Реальные классы (True Labels)")
plt.xlabel("Предсказанные классы (Predicted Labels)")
plt.tight_layout()
plt.show()